# L09 · NB 01 — Monday morning: search is broken

> *Marcus forwards an angry email at 9:14am. A customer typed "blue summer dress" into NorthStar's search bar. We have ten dresses that fit. None of them came up. The customer left.*

Sarah's first move is to reproduce the problem. Open the catalogue, run the same query through the current keyword search, see what happens.

This notebook is pre-class. Run it before class so you walk in with first-hand experience of *what's actually broken*.

## 1 · Load the catalogue

In [1]:
import pandas as pd

# Load data — local file if present (VS Code), else fetch from GitHub (works in Colab).
import os
_LOCAL = 'data/northstar_catalogue.csv'
_URL = 'https://raw.githubusercontent.com/flexfengfeng/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/northstar_catalogue.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)
print(f"Products in catalogue: {len(df)}")
print(f"Categories: {df['category'].unique().tolist()}")
print(f"\nFirst three rows:")
df.head(3)

Products in catalogue: 76
Categories: ['dress', 'shirt', 'trouser', 'coat', 'knit', 'footwear', 'accessory', 'activewear', 'home', 'kids']

First three rows:


,product_id,name,category,description,price_gbp
0,P0001,Lila Floral Sundress,dress,Lightweight floral frock perfect for warm summ...,49
1,P0002,Marina Wrap Dress,dress,Elegant wrap-style dress in deep navy. Suitabl...,89
2,P0003,Cassia Maxi Gown,dress,Flowing full-length gown with adjustable strap...,65


## 2 · Eyeball the dresses

The customer searched for "blue summer dress". Let's look at all the dresses we have.

In [2]:
dresses = df[df['category'] == 'dress'].reset_index(drop=True)
print(f"We have {len(dresses)} dresses.")
print()
for _, row in dresses.iterrows():
    print(f"  [{row['product_id']}]  {row['name']}")
    print(f"     {row['description']}")
    print()

We have 10 dresses.

  [P0001]  Lila Floral Sundress
     Lightweight floral frock perfect for warm summer days. Tea-length, pastel print, breathable cotton.

  [P0002]  Marina Wrap Dress
     Elegant wrap-style dress in deep navy. Suitable for office or evening occasions.

  [P0003]  Cassia Maxi Gown
     Flowing full-length gown with adjustable straps. Beach holiday essential.

  [P0004]  Holly Knit Jumper Dress
     Cosy ribbed knit dress in oatmeal. Perfect winter layer with thick tights.

  [P0005]  Ivy Slip Dress
     Minimalist satin slip in champagne. Layer over a tee or wear alone for evenings.

  [P0006]  Bea Tea-Length Frock
     Vintage-inspired knee-length number in dusty rose. Spring wedding-guest favourite.

  [P0007]  Sienna Bodycon Dress
     Fitted stretch jersey in midnight black. Goes from desk to drinks effortlessly.

  [P0008]  Aurora Pleated Midi
     Romantic pleated midi in soft lavender. Twirl-friendly and waist-defining.

  [P0009]  Petal Smock Dress
     Rel

Note what you DON'T see in those descriptions: very few of them contain all the words "blue", "summer", and "dress" together. The catalogue uses words like "frock", "sundress", "gown", and refers to colour in indirect ways (pastel, navy, champagne, etc).

A keyword-matching search will struggle here. Let's confirm.

## 3 · The naive keyword search

The simplest possible search: split text into lowercase words, return any product whose description contains AT LEAST ONE query word.

In [3]:
import re

def tokens(text):
    """Lowercase + split on non-word characters. Returns a set."""
    return set(re.findall(r'\w+', text.lower()))

def keyword_search(query, df, score_fn=None):
    """Return df with a 'score' column = count of query words in description+name."""
    qtoks = tokens(query)
    results = df.copy()
    results['score'] = results.apply(
        lambda r: len(qtoks & tokens(r['description'] + ' ' + r['name'])), axis=1
    )
    return results.sort_values('score', ascending=False)

query = "blue summer dress"
print(f"Query: {query!r}")
print()
results = keyword_search(query, df).head(10)
print(results[['product_id','category','name','score']].to_string(index=False))

Query: 'blue summer dress'

product_id category                    name  score
     P0001    dress    Lila Floral Sundress      1
     P0004    dress Holly Knit Jumper Dress      1
     P0005    dress          Ivy Slip Dress      1
     P0007    dress    Sienna Bodycon Dress      1
     P0002    dress       Marina Wrap Dress      1
     P0009    dress       Petal Smock Dress      1
     P0010    dress    Marigold Shift Dress      1
     P0047 footwear Sand Espadrille Sandals      1
     P0037     knit    Cobalt V-Neck Jumper      1
     P0014    shirt       Frost Linen Shirt      1


**Observe:** What got returned?

You should see the search returning anything that contains the word "summer" or "blue" or "dress" — but several real dresses (Lila, Cassia, Petal) score 0 or 1 because their descriptions don't use those exact words.

Worse: items in OTHER categories (a yoga top with the word "blue" in it, a "Bumblebee Romper" with "yellow and black") may score equal to or higher than relevant dresses.

## 4 · Inspect the misses

Which dresses got zero score for the query? These are the actual failures Marcus is hearing about.

In [4]:
dresses_with_score = keyword_search(query, dresses)
zero_score = dresses_with_score[dresses_with_score['score'] == 0]
print(f"{len(zero_score)} of {len(dresses)} dresses got ZERO keyword overlap with 'blue summer dress':")
print()
for _, row in zero_score.iterrows():
    print(f"  [{row['product_id']}]  {row['name']}")
    print(f"     description: {row['description']}")
    # what words from query SHOULD have matched?
    print(f"     description uses these instead of 'dress':", end=' ')
    syns = [w for w in ['frock','sundress','gown','sundress','smock','shift','slip','jumper'] if w in row['description'].lower()]
    print(', '.join(syns) if syns else '(no obvious synonyms)')
    print()

3 of 10 dresses got ZERO keyword overlap with 'blue summer dress':

  [P0003]  Cassia Maxi Gown
     description: Flowing full-length gown with adjustable straps. Beach holiday essential.
     description uses these instead of 'dress': gown

  [P0006]  Bea Tea-Length Frock
     description: Vintage-inspired knee-length number in dusty rose. Spring wedding-guest favourite.
     description uses these instead of 'dress': (no obvious synonyms)

  [P0008]  Aurora Pleated Midi
     description: Romantic pleated midi in soft lavender. Twirl-friendly and waist-defining.
     description uses these instead of 'dress': (no obvious synonyms)



That's the customer-experience problem in one screen.

The Lila Floral Sundress is exactly what the customer wanted. The description says *"lightweight floral frock perfect for warm summer days"*. To a keyword matcher, **frock ≠ dress**. Match score: 1 (just "summer"). The product is buried.

## 5 · Will bandaids help? Stop-words + stemming

In [5]:
# Stem each word to its root: 'running' -> 'run', 'dresses' -> 'dress'
# We'll use a tiny hand-rolled stemmer to keep deps zero.
STOPWORDS = set('the a an of for in on with and or but is are to'.split())

def normalize(text):
    out = []
    for w in re.findall(r'\w+', text.lower()):
        if w in STOPWORDS: continue
        # Strip simple plural / participle endings
        for suf in ('ies', 'es', 'ed', 'ing', 's'):
            if w.endswith(suf) and len(w) - len(suf) >= 3:
                w = w[:-len(suf)]; break
        out.append(w)
    return set(out)

def keyword_search_v2(query, df):
    qtoks = normalize(query)
    res = df.copy()
    res['score'] = res.apply(
        lambda r: len(qtoks & normalize(r['description'] + ' ' + r['name'])), axis=1
    )
    return res.sort_values('score', ascending=False)

print("v2 search (stemming + stop-words):")
print()
res_v2 = keyword_search_v2(query, df).head(10)
print(res_v2[['product_id','category','name','score']].to_string(index=False))

v2 search (stemming + stop-words):

product_id category                    name  score
     P0001    dress    Lila Floral Sundress      1
     P0004    dress Holly Knit Jumper Dress      1
     P0005    dress          Ivy Slip Dress      1
     P0007    dress    Sienna Bodycon Dress      1
     P0002    dress       Marina Wrap Dress      1
     P0009    dress       Petal Smock Dress      1
     P0010    dress    Marigold Shift Dress      1
     P0047 footwear Sand Espadrille Sandals      1
     P0037     knit    Cobalt V-Neck Jumper      1
     P0014    shirt       Frost Linen Shirt      1


Marginally better — but the fundamental problem remains. Stemming reduces "dresses" → "dress", but doesn't know that *frock* → *dress*. No amount of lexical processing helps because **the synonym problem isn't lexical, it's semantic**.

## 6 · Quantify the damage across realistic queries

Let's measure how often keyword search returns NOTHING USEFUL across a small set of plausible customer queries.

In [6]:
queries = [
    "blue summer dress",
    "warm winter jumper",
    "smart office shoes",
    "lightweight rain jacket",
    "cosy throw for sofa",
    "gym leggings",
    "beach holiday outfit",
    "running trainers",
]

print(f"{'Query':30s} {'Top-1 match':45s} {'Score':>6s}")
print("-" * 85)
for q in queries:
    res = keyword_search_v2(q, df).iloc[0]
    print(f"{q:30s} {res['name']:45s} {res['score']:>6d}")

Query                          Top-1 match                                    Score
-------------------------------------------------------------------------------------
blue summer dress              Lila Floral Sundress                               1
warm winter jumper             Holly Knit Jumper Dress                            2
smart office shoes             Cloud Running Trainers                             1
lightweight rain jacket        Ember Quilted Jacket                               2
cosy throw for sofa            Wool Throw Blanket                                 2
gym leggings                   Storm Yoga Leggings                                1
beach holiday outfit           Cassia Maxi Gown                                   2
running trainers               Cloud Running Trainers                             2


Notice the pattern: for queries where the obvious keyword appears in the catalogue (like "leggings" — we have *Storm Yoga Leggings*), top-1 is great. For queries that depend on synonyms or paraphrasing ("warm winter jumper", "cosy throw"), top-1 is often a near-miss or a complete dud.

Real customer queries skew **strongly** toward the second kind. That's what's hurting us.

## 7 · What would success look like?

For each of the queries above, write down the ONE product from the catalogue you'd want as top-1. We'll come back to this list in NB 03 and grade the semantic-search model against it.

In [7]:
# Sarah's ground truth — what SHOULD top-1 be for each query?
# We'll use this as a benchmark in NB 03.
ground_truth = {
    "blue summer dress":         "Lila Floral Sundress",   # blue not in desc, but lightest dress
    "warm winter jumper":        "Roan Cable Jumper",      # chunky wool jumper
    "smart office shoes":        "Loam Ankle Boots",       # office-to-evening leather
    "lightweight rain jacket":   "Trail Windbreaker",      # packable shower-proof
    "cosy throw for sofa":       "Wool Throw Blanket",     # sofa-side warmth
    "gym leggings":              "Storm Yoga Leggings",    # exact match — easy
    "beach holiday outfit":      "Cassia Maxi Gown",       # beach essential
    "running trainers":          "Cloud Running Trainers", # exact match — easy
}

# Save for next notebook
import json
with open('ground_truth.json', 'w') as f:
    json.dump(ground_truth, f, indent=2)
print('Saved ground-truth queries to ground_truth.json')

# How many does keyword search get right?
correct_kw = 0
for q, expected in ground_truth.items():
    top1 = keyword_search_v2(q, df).iloc[0]['name']
    correct = top1 == expected
    correct_kw += int(correct)
    mark = '✅' if correct else '❌'
    print(f"  {mark}  {q!r:30s} got: {top1!r}")

print(f"\nKeyword search top-1 accuracy: {correct_kw}/{len(ground_truth)} = {correct_kw/len(ground_truth):.0%}")

Saved ground-truth queries to ground_truth.json
  ✅  'blue summer dress'            got: 'Lila Floral Sundress'
  ❌  'warm winter jumper'           got: 'Holly Knit Jumper Dress'
  ❌  'smart office shoes'           got: 'Cloud Running Trainers'
  ❌  'lightweight rain jacket'      got: 'Ember Quilted Jacket'
  ✅  'cosy throw for sofa'          got: 'Wool Throw Blanket'
  ✅  'gym leggings'                 got: 'Storm Yoga Leggings'
  ✅  'beach holiday outfit'         got: 'Cassia Maxi Gown'
  ✅  'running trainers'             got: 'Cloud Running Trainers'

Keyword search top-1 accuracy: 5/8 = 62%


**Today's baseline.** Tomorrow we'll beat this with a model that has actually read English.

## 8 · Three thought-questions for class

1. Without using the word *dress*, *frock*, *gown*, *sundress*, or *outfit* — write three different ways a customer might describe what they want when they search for a summer dress.
2. The query *gym leggings* worked perfectly because the product is called *Storm Yoga Leggings*. What would happen if the product was just called *Storm Leggings* with a description focused on construction ("high-rise compression")? Would keyword search still find it?
3. If you had to bolt one feature onto keyword search without doing any ML, what would you build? Why does it not fully solve the problem?

Bring your guesses to class.